# Odia Heritage Cloze - Difficulty Assignment

Assigns `difficulty` (Easy / Medium / Hard) to the 95 rows of
`heritage_factual_short_answer.jsonl`.

**Task.** Cloze-style factual short answer. `question` holds a place name and a
sentence with one span blanked out; `answer` is that span, verbatim from the
source passage (a distance, a count, a height). The full source paragraph is in
`explanation`.

**Pipeline - unchanged from the NCERT notebook** except for how answers are
scored:

1. Three models answer each question: **Mistral-7B**, **Llama-3.1-8B**,
   **Gemma-2-9B**. Same 4-bit loading, Drive caching, batching and resume.
2. Each answer is scored by **exact match** against the gold span - not
   ROUGE-L, and no LLM judge. The answers are short verbatim spans, so exact
   match is both the right metric and the file's declared `eval_metric`.
3. The three 1/0 results sum into the usual rule:

| Models correct | Difficulty |
|---|---|
| 3 / 3 | Easy |
| 2 / 3 | Medium |
| 0-1 / 3 | Hard |

**What changed and why.** Exact match is already binary, so the judge, the
rubric and the whole threshold-discovery step are gone - there is nothing to
threshold. That removes one model download (~9 GB) and one cell pair. The
question/answer content is used exactly as generated; nothing is regenerated,
rebalanced or filtered.

**Expect a Hard skew.** These are specific numeric facts about Odia heritage
sites asked closed-book - almost no model knows that Satkosia is 97 km from
Anugul. That is a real property of the task, not a bug, and the notebook does
not compensate for it. `OPEN_BOOK = True` in Cell 3 supplies the source
paragraph as context if you want the reading-comprehension variant instead.

**Output.** `heritage_difficulty.jsonl` - all 14 schema fields with
`difficulty` filled in, plus an audit file holding every model answer.

### Cell 1 - Install dependencies and authenticate

**An HF token is required.** Llama-3.1 and Gemma-2 are gated. Mistral and
Qwen3 are not. Accept the two licences on huggingface.co, create a **read**
token, then add it in Colab via the **key icon** as a secret named `HF_TOKEN`.
Use the secret rather than pasting the token into a cell.

In [1]:
!pip -q install -U transformers accelerate bitsandbytes huggingface_hub

HF_OK = False
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    HF_OK = True
    print("HF login OK")
except Exception as e:
    print("No HF token ({}: {})".format(type(e).__name__, e))
    print("Mistral and Qwen3 still work; gated Llama/Gemma will fail with a 401.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 35.2 MB/s eta 0:00:00
HF login OK


### Cell 2 - Mount Drive

Drive is the weight cache: each model is downloaded once, quantised to 4-bit,
and saved here, so later runs skip the download entirely. There is **no judge model here**, so
only the three generators are cached - about **16.5 GB**, versus ~25 GB for the
NCERT notebook. If you already ran that one, these three are cached and this
notebook downloads nothing.

| Model | 4-bit on Drive |
|---|---|
| Mistral-7B | ~4.5 GB |
| Llama-3.1-8B | ~5.5 GB |
| Gemma-2-9B | ~6.5 GB |
| **Total** | **~16.5 GB** |

In [2]:
import os

DRIVE_OK    = False
DRIVE_MOUNT = "/drive"
CACHE_DIR   = os.path.join(DRIVE_MOUNT, "MyDrive", "models")

try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT, force_remount=True)
    DRIVE_OK = os.path.isdir(os.path.join(DRIVE_MOUNT, "MyDrive"))
except ImportError:
    print("Not running on Colab - Drive caching disabled.")
except Exception as e:
    print("DRIVE MOUNT FAILED: {}".format(e))

if DRIVE_OK:
    os.makedirs(CACHE_DIR, exist_ok=True)
    print("Drive mounted | weight cache: {}".format(CACHE_DIR))
    try:
        st = os.statvfs(DRIVE_MOUNT)
        free = st.f_bavail * st.f_frsize / 1e9
        print("free on Drive: {:.1f} GB".format(free))
        if free < 25:
            print("  NOTE: caching all four models needs ~25 GB. You have less,")
            print("  so some will re-download each session. That still works.")
    except Exception:
        pass
else:
    print("\nWARNING: no Drive - nothing is cached between sessions.")

Mounted at /drive
Drive mounted | weight cache: /drive/MyDrive/models
free on Drive: 66.5 GB


### Cell 3 - Configuration

- `GENERATORS` - the same three models as every other split. **No judge**:
  exact match needs no model, so `JUDGE`, `RUBRIC` and the threshold settings
  are gone.
- `OPEN_BOOK` - `False` (default) asks the question closed-book, testing
  whether a model *knows* the fact. `True` supplies the source paragraph from
  `explanation` as context, turning it into reading comprehension. Closed-book
  will skew Hard; that is expected and is not corrected for.
- `N_ROWS` - the file has 95 rows; leaving this at or above 95 scores all of
  them rather than sampling.
- `eval_metric` is left as the file's own `accuracy` - it already describes
  what this notebook does.

In [3]:
import gc
import re
import json
import random
import shutil
import statistics
from collections import Counter

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ---- paths ----
INPUT_FILE  = "heritage_factual_short_answer.jsonl"
OUTPUT_FILE = "heritage_difficulty.jsonl"
AUDIT_FILE  = "heritage_audit.jsonl"
GEN_DIR     = "gen_progress"       # one file per generator

# ---- sampling ----
N_ROWS = 95                        # the whole file; no sampling
SEED   = 42

# ---- task setting ----
OPEN_BOOK = False                  # True -> supply the source paragraph

# ---- generation ----
MAX_NEW_TOKENS = 32                # the answer is a short span
BATCH_SIZE     = 25

# ---- schema ----
SET_EVAL_METRIC = None             # keep the file's own "accuracy"

# ---- models ----
GENERATORS = [
    {"name": "mistral", "repo": "mistralai/Mistral-7B-Instruct-v0.3"},   # ungated
    {"name": "llama",   "repo": "meta-llama/Llama-3.1-8B-Instruct"},     # GATED
    {"name": "gemma",   "repo": "google/gemma-2-9b-it", "attn": "eager"},# GATED
]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,     # T4 has no bf16
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

SCHEMA_KEYS = [
    "id", "source", "category", "subcategory", "region", "language",
    "difficulty", "task_type", "question", "options", "answer",
    "explanation", "cultural_attr", "eval_metric",
]

os.makedirs(GEN_DIR, exist_ok=True)

print("Generators (no judge - exact match needs no model):")
for m in GENERATORS:
    cached = DRIVE_OK and os.path.isfile(
        os.path.join(CACHE_DIR, m["name"] + "_4bit", "config.json"))
    print("  {:<11} {:<40} {}".format(
        m["name"], m["repo"], "cached" if cached else "will download"))
print("setting: {}".format("OPEN BOOK (paragraph given)" if OPEN_BOOK
                           else "CLOSED BOOK (no context)"))
print("\ndevice:", "cuda" if torch.cuda.is_available() else "CPU (will be very slow)")

Generators (no judge - exact match needs no model):
  mistral     mistralai/Mistral-7B-Instruct-v0.3       cached
  llama       meta-llama/Llama-3.1-8B-Instruct         cached
  gemma       google/gemma-2-9b-it                     cached
setting: CLOSED BOOK (no context)

device: cuda


### Cell 4 - Load

The file is only 95 rows, so with `N_ROWS = 95` every row is scored - there is
no sampling and no filtering. Questions and answers are used exactly as
generated; nothing here rewrites them.

The printed breakdown shows how many distinct heritage sites are covered and
what the gold spans look like, which is worth a glance before spending GPU
time.

In [5]:
with open(INPUT_FILE, encoding="utf-8") as f:
    all_rows = [json.loads(line) for line in f]
print("Loaded {} rows from {}".format(len(all_rows), INPUT_FILE))


def usable(row):
    return (str(row.get("question") or "").strip()
            and str(row.get("answer") or "").strip())


pool = [r for r in all_rows if usable(r)]
if len(pool) < len(all_rows):
    print("dropped {} rows with an empty question or answer".format(
        len(all_rows) - len(pool)))

if N_ROWS >= len(pool):
    sample = pool                      # score everything, no sampling
    print("scoring ALL {} rows".format(len(sample)))
else:
    random.seed(SEED)
    sample = random.sample(pool, N_ROWS)
    print("sampled {} of {} rows".format(len(sample), len(pool)))

places = Counter(str(r["question"]).split("\n")[0] for r in sample)
alen   = sorted(len(str(r["answer"]).split()) for r in sample)
print("  heritage sites covered : {}".format(len(places)))
print("  gold answer words      : min {}, median {}, max {}".format(
    alen[0], alen[len(alen) // 2], alen[-1]))
print("  language               : {}".format(
    dict(Counter(r["language"] for r in sample))))

print("\n--- example row ---")
q = str(sample[0]["question"]).split("\n")
print("  place : {}".format(q[0]))
print("  cloze : {}".format(" ".join(q[-1].split())[:96]))
print("  gold  : {!r}".format(sample[0]["answer"]))

Loaded 95 rows from heritage_factual_short_answer.jsonl
scoring ALL 95 rows
  heritage sites covered : 34
  gold answer words      : min 1, median 1, max 2
  language               : {'or': 95}

--- example row ---
  place : ପ୍ରକୃତିପ୍ରେମୀଙ୍କ ସ୍ୱର୍ଗ ସାତକୋଶିଆ
  cloze : ଅନୁଗୁଳଠାରୁ ସାତକୋଶିଆର ଦୂରତ୍ୱ ପ୍ରାୟ ______
  gold  : '୯୭ କି:ମି'


### Cell 5 - Cloze prompt

The task is fill-in-the-blank, so the prompt asks for **only the missing span**
- not a sentence, not an explanation. That matters for exact match: any extra
word makes the answer wrong, so the instruction has to be unambiguous and
`MAX_NEW_TOKENS` is small.

**No few-shot examples.** Every row in this file is scored, so any example
drawn from it would leak that row's gold answer. Rather than hold rows out of
the evaluation, the prompt is zero-shot with an explicit format instruction.

`OPEN_BOOK` decides whether the source paragraph from `explanation` is supplied
as context. Closed-book asks whether the model knows the fact; open-book asks
whether it can read Odia and extract a span.

In [6]:
GEN_INSTRUCTIONS = (
    "You are completing fill-in-the-blank questions about heritage and tourist "
    "sites in Odisha, India. The text is in Odia.\n\n"
    "Each question contains a blank marked ______ . Reply with ONLY the text "
    "that belongs in the blank - usually a number with its unit, such as a "
    "distance or a count.\n\n"
    "Do not repeat the sentence. Do not explain. Output the missing text and "
    "nothing else."
)

BLANK = "______"


def flat(text):
    return " ".join(str(text).split())


def build_query(row):
    q = str(row["question"]).split("\n")
    place, cloze = q[0].strip(), flat(q[-1])
    parts = []
    if OPEN_BOOK and str(row.get("explanation") or "").strip():
        parts.append("Passage:\n" + flat(row["explanation"]))
    parts.append("Place: " + place)
    parts.append("Sentence: " + cloze)
    parts.append("Missing text:")
    return "\n\n".join(parts)


def build_completion(row):
    return GEN_INSTRUCTIONS + "\n\n" + build_query(row)


def build_chat_messages(row):
    return [{"role": "system", "content": GEN_INSTRUCTIONS},
            {"role": "user",   "content": build_query(row)}]


print("=" * 66)
print(build_completion(sample[0]))
print("=" * 66)
print("[gold: {!r}]".format(sample[0]["answer"]))

You are completing fill-in-the-blank questions about heritage and tourist sites in Odisha, India. The text is in Odia.

Each question contains a blank marked ______ . Reply with ONLY the text that belongs in the blank - usually a number with its unit, such as a distance or a count.

Do not repeat the sentence. Do not explain. Output the missing text and nothing else.

Place: ପ୍ରକୃତିପ୍ରେମୀଙ୍କ ସ୍ୱର୍ଗ ସାତକୋଶିଆ

Sentence: ଅନୁଗୁଳଠାରୁ ସାତକୋଶିଆର ଦୂରତ୍ୱ ପ୍ରାୟ ______

Missing text:
[gold: '୯୭ କି:ମି']


### Cell 6 - Shared model loading and answer generation

`load_model` implements download-once for **every** model including the judge:
if `models/<name>_4bit` exists in Drive it is loaded directly (already 4-bit, so
passing a fresh `BitsAndBytesConfig` would conflict and is omitted); otherwise
the repo is downloaded, quantised, and saved to Drive.

`trust_remote_code` stays off - repo-shipped modelling code is often written
against an older transformers API.

`clean_answer` keeps only the first line and strips the labels models prepend
(`Missing text:`, `Answer:`, `Sure, ...`) plus surrounding quotes. Under exact
match any leftover boilerplate makes a correct answer count as wrong, so this
is doing more work here than in the NCERT notebook - Cell 9 reports how often
a model's output merely *contains* the gold span, which is the signal that
cleaning is losing matches.

In [7]:
def prompt_style(tokenizer):
    # base checkpoints have no chat template at all
    return "chat" if getattr(tokenizer, "chat_template", None) else "completion"


def cache_path(spec):
    return os.path.join(CACHE_DIR, spec["name"] + "_4bit")


def load_model(spec):
    cached     = cache_path(spec)
    from_drive = DRIVE_OK and os.path.isfile(os.path.join(cached, "config.json"))
    source     = cached if from_drive else spec["repo"]

    kwargs = {"device_map": "auto", "trust_remote_code": False}
    if from_drive:
        how = "Drive cache (already 4-bit)"
    else:
        kwargs["quantization_config"] = bnb_config
        how = "HuggingFace download -> 4-bit"
    if spec.get("attn"):
        kwargs["attn_implementation"] = spec["attn"]
        how += ", attn=" + spec["attn"]

    print("  loading {} [{}]".format(source, how))
    tokenizer = AutoTokenizer.from_pretrained(source)
    model = AutoModelForCausalLM.from_pretrained(source, **kwargs).eval()

    if not from_drive and DRIVE_OK:
        print("  saving 4-bit copy to {} (one time)...".format(cached))
        os.makedirs(cached, exist_ok=True)
        model.save_pretrained(cached)
        tokenizer.save_pretrained(cached)
        print("  saved - future runs skip the download")

    print("  ready | VRAM: {:.2f}GB | prompt style: {}".format(
        torch.cuda.memory_allocated() / 1e9, prompt_style(tokenizer)))
    return model, tokenizer


def unload(model, tokenizer):
    del model, tokenizer
    shutil.rmtree("/root/.cache/huggingface/hub/", ignore_errors=True)
    gc.collect()
    torch.cuda.empty_cache()


_LABEL = re.compile(
    r"^\s*(sure[,!]?\s*)?(the\s+)?(missing\s+text|answer|blank)\s*[:\-]\s*",
    re.I)


def clean_answer(text):
    t = (text or "").strip()
    t = re.split(r"\n\s*(?:Sentence|Place|Passage)\s*:", t)[0]
    for line in t.split("\n"):
        line = _LABEL.sub("", line.strip()).strip().strip('"\u2018\u2019\u201c\u201d')
        if line:
            return " ".join(line.split())
    return ""


@torch.no_grad()
def answer_question(model, tokenizer, row):
    if prompt_style(tokenizer) == "completion":
        text = build_completion(row)
    else:
        msgs = build_chat_messages(row)
        try:
            text = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True)
        except Exception:
            # some templates (Gemma) reject a system role - fold it into the
            # first user turn rather than dropping the instructions
            merged = [dict(m) for m in msgs[1:]]
            merged[0]["content"] = msgs[0]["content"] + "\n\n" + merged[0]["content"]
            text = tokenizer.apply_chat_template(
                merged, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    n_in   = inputs["input_ids"].shape[1]
    out = model.generate(**inputs,
                         max_new_tokens=MAX_NEW_TOKENS,
                         do_sample=False,
                         pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    return clean_answer(tokenizer.decode(out[0][n_in:], skip_special_tokens=True))


for raw, want in [
    ("Missing text: 97 km", "97 km"),
    ("Answer: 22", "22"),
    ('"5"', "5"),
    ("97 km\nSentence: next one", "97 km"),
    ("", ""),
]:
    got = clean_answer(raw)
    assert got == want, (raw, got, want)
print("Loading and generation functions defined")

Loading and generation functions defined


### Cell 7 - Run the three generators

Each generator answers all 200 questions, one model at a time - loaded, run,
unloaded - so peak VRAM stays near 6 GB. Every finished batch is appended to
`gen_progress/<model>.jsonl` before the next begins, so a disconnect costs at
most `BATCH_SIZE` rows and a completed model is skipped without loading.

The long cell: roughly **10-15 min per generator**, plus downloads on the first
run.

In [8]:
def run_generator(spec, rows):
    prog = os.path.join(GEN_DIR, spec["name"] + ".jsonl")

    done = {}
    if os.path.exists(prog):
        with open(prog, encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                done[item["id"]] = item
        print("  resuming - {}/{} already answered".format(len(done), len(rows)))

    remaining = [r for r in rows if r["id"] not in done]
    if not remaining:
        print("  {} already complete - skipping load".format(spec["name"]))
        return done

    model, tokenizer = load_model(spec)
    total_batches = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE

    for start in range(0, len(remaining), BATCH_SIZE):
        batch = remaining[start:start + BATCH_SIZE]
        results = []
        for row in batch:
            ans = answer_question(model, tokenizer, row)
            results.append({"id": row["id"], "answer": ans,
                            "n_words": len(ans.split())})
        with open(prog, "a", encoding="utf-8") as f:
            for item in results:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
        done.update({i["id"]: i for i in results})
        print("  batch {}/{} saved - {}/{} rows".format(
            start // BATCH_SIZE + 1, total_batches, len(done), len(rows)))

    unload(model, tokenizer)
    print("  {} complete".format(spec["name"]))
    return done


answers = {}
for spec in GENERATORS:
    print("\n=== {} ===".format(spec["name"]))
    answers[spec["name"]] = run_generator(spec, sample)

print("\nAll generators done")


=== mistral ===
  loading /drive/MyDrive/models/mistral_4bit [Drive cache (already 4-bit)]


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  ready | VRAM: 4.14GB | prompt style: chat
  batch 1/4 saved - 25/95 rows
  batch 2/4 saved - 50/95 rows
  batch 3/4 saved - 75/95 rows
  batch 4/4 saved - 95/95 rows
  mistral complete

=== llama ===
  loading /drive/MyDrive/models/llama_4bit [Drive cache (already 4-bit)]


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  ready | VRAM: 9.31GB | prompt style: chat


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  batch 1/4 saved - 25/95 rows
  batch 2/4 saved - 50/95 rows
  batch 3/4 saved - 75/95 rows
  batch 4/4 saved - 95/95 rows
  llama complete

=== gemma ===
  loading /drive/MyDrive/models/gemma_4bit [Drive cache (already 4-bit), attn=eager]


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

  ready | VRAM: 6.14GB | prompt style: chat
  batch 1/4 saved - 25/95 rows
  batch 2/4 saved - 50/95 rows
  batch 3/4 saved - 75/95 rows
  batch 4/4 saved - 95/95 rows
  gemma complete

All generators done


### Cell 8 - Exact-match scoring

This replaces the NCERT notebook's LLM judge. The gold answers here are short
verbatim spans, so exact match is both the right metric and the one the file
declares in `eval_metric`.

`normalize` collapses whitespace and strips trailing punctuation (including the
Odia danda) and surrounding quotes before comparing - the standard exact-match
normalisation. It does **not** touch digits, units or word order, so a wrong
number or a missing unit still counts as wrong.

Two further figures are recorded for diagnosis but **never used for the vote**:

- `em_loose` - equality after removing all whitespace, which catches spacing
  differences inside a unit like `୯୭ କି:ମି`.
- `contains` - whether the gold span appears anywhere in the model's output.

If `contains` is far above `em`, the models know the fact but are wrapping it
in extra words, and the prompt or `clean_answer` needs work - not the data.

In [9]:
_TRAIL = re.compile(r"[\s\u0964\.,;:!\?\)\]]+$")
_LEAD  = re.compile(r"^[\s\(\[]+")


def normalize(text):
    t = " ".join(str(text or "").split())
    t = t.strip('"\u2018\u2019\u201c\u201d')
    t = _LEAD.sub("", t)
    t = _TRAIL.sub("", t)
    return t


def score_answer(pred, gold):
    p, g = normalize(pred), normalize(gold)
    return {
        "em":       int(bool(g) and p == g),
        "em_loose": int(bool(g) and "".join(p.split()) == "".join(g.split())),
        "contains": int(bool(g) and g in p),
    }


for pred, gold, want in [
    ("\u0b6f\u0b6d \u0b15\u0b3f:\u0b2e\u0b3f", "\u0b6f\u0b6d \u0b15\u0b3f:\u0b2e\u0b3f", 1),
    ("\u0b6f\u0b6d \u0b15\u0b3f:\u0b2e\u0b3f \u0964", "\u0b6f\u0b6d \u0b15\u0b3f:\u0b2e\u0b3f", 1),
    ('"\u0b6b\u0b1f\u0b3f"', "\u0b6b\u0b1f\u0b3f", 1),
    ("\u0b6c\u0b6d \u0b15\u0b3f:\u0b2e\u0b3f", "\u0b6f\u0b6d \u0b15\u0b3f:\u0b2e\u0b3f", 0),
    ("\u0b6f\u0b6d", "\u0b6f\u0b6d \u0b15\u0b3f:\u0b2e\u0b3f", 0),
    ("", "\u0b6f\u0b6d", 0),
]:
    got = score_answer(pred, gold)["em"]
    assert got == want, (pred, gold, got, want)
print("Exact-match scoring defined and checked")

Exact-match scoring defined and checked


### Cell 9 - Score every answer

No model is loaded here - exact match is pure string comparison, so all
3 x 95 comparisons run instantly and there is nothing to resume.

The printout is the first place a problem would show. Watch the gap between
**EM** and **contains**: if a model's `contains` is much higher, it knows the
answers but is not obeying the output format, and that is a prompting problem
rather than a difficulty signal.

In [10]:
scores = {}
for row in sample:
    for spec in GENERATORS:
        pred = answers[spec["name"]][row["id"]]["answer"]
        scores[(row["id"], spec["name"])] = score_answer(pred, row["answer"])

print("comparisons: {}\n".format(len(scores)))
print("  {:<10} {:>8} {:>10} {:>10} {:>8}".format(
    "model", "EM", "EM(loose)", "contains", "empty"))
for spec in GENERATORS:
    v = [scores[(r["id"], spec["name"])] for r in sample]
    empty = sum(1 for r in sample
                if not answers[spec["name"]][r["id"]]["answer"].strip())
    n = len(v)
    print("  {:<10} {:>7.1%} {:>10.1%} {:>10.1%} {:>8}".format(
        spec["name"],
        sum(x["em"] for x in v) / n,
        sum(x["em_loose"] for x in v) / n,
        sum(x["contains"] for x in v) / n,
        empty))

gap = max((sum(scores[(r["id"], s["name"])]["contains"] for r in sample)
           - sum(scores[(r["id"], s["name"])]["em"] for r in sample))
          for s in GENERATORS)
if gap > 0.15 * len(sample):
    print("\n  NOTE: 'contains' exceeds EM by a wide margin for at least one")
    print("  model - it is producing the right span wrapped in extra words.")
    print("  That is a formatting issue, not difficulty. Check clean_answer.")

comparisons: 285

  model            EM  EM(loose)   contains    empty
  mistral       0.0%       0.0%       0.0%        0
  llama         0.0%       0.0%       0.0%        0
  gemma         1.1%       1.1%       1.1%        0


### Cell 10 - Assign difficulty and write the schema

Each model contributes 1 if its answer exactly matches the gold span, and the
three votes sum into Easy / Medium / Hard - the same rule as every other split.
There is no threshold to choose: exact match is already binary.

Output rows are rebuilt key-by-key from `SCHEMA_KEYS`, so all 14 fields are
preserved in schema order. `difficulty` is the only value that changes -
`eval_metric` keeps the file's own `accuracy`, and the questions and answers
are passed through untouched.

In [11]:
def get_difficulty(votes):
    score = sum(votes)
    if score == 3:
        return "Easy"
    elif score == 2:
        return "Medium"
    else:
        return "Hard"


final_results, audit = [], []

for row in sample:
    votes = [scores[(row["id"], s["name"])]["em"] for s in GENERATORS]
    difficulty = get_difficulty(votes)

    enriched = {**row, "difficulty": difficulty}
    if SET_EVAL_METRIC:
        enriched["eval_metric"] = SET_EVAL_METRIC
    final_results.append({k: enriched.get(k) for k in SCHEMA_KEYS})

    audit.append({
        "id":          row["id"],
        "difficulty":  difficulty,
        "votes":       votes,
        "gold":        row["answer"],
        "open_book":   OPEN_BOOK,
        "place":       str(row["question"]).split("\n")[0],
        "cloze":       " ".join(str(row["question"]).split("\n")[-1].split()),
        "predictions": {s["name"]: answers[s["name"]][row["id"]]["answer"]
                        for s in GENERATORS},
        "detail":      {s["name"]: scores[(row["id"], s["name"])]
                        for s in GENERATORS},
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for item in final_results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
with open(AUDIT_FILE, "w", encoding="utf-8") as f:
    for item in audit:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved -> {} ({} rows)".format(OUTPUT_FILE, len(final_results)))
print("Audit -> {}".format(AUDIT_FILE))

Saved -> heritage_difficulty.jsonl (95 rows)
Audit -> heritage_audit.jsonl


### Cell 11 - Verify and report

1. **Schema** - all 14 fields in order, no nulls left in `difficulty`, and the
   questions and answers identical to the input file.
2. **Difficulty distribution**, reported as it falls. A heavy Hard skew is the
   expected outcome closed-book: these are specific numeric facts about
   individual Odia heritage sites. Nothing here rebalances it.
3. **Per-model exact match**, with `contains` alongside so a formatting failure
   is distinguishable from genuine difficulty.
4. **Distribution across heritage sites**, so you can see whether Hard rows
   cluster on a few places or spread evenly.

In [12]:
bad_keys = [r["id"] for r in final_results if list(r.keys()) != SCHEMA_KEYS]
missing  = [r["id"] for r in final_results if r["difficulty"] is None]
src_by_id = {r["id"]: r for r in all_rows}
altered = [r["id"] for r in final_results
           if r["question"] != src_by_id[r["id"]]["question"]
           or r["answer"] != src_by_id[r["id"]]["answer"]]
print("Schema check : {} rows | wrong keys: {} | null difficulty: {}".format(
    len(final_results), len(bad_keys), len(missing)))
print("questions/answers altered vs input: {}".format(len(altered)))
print("eval_metric  : {}".format(
    dict(Counter(r["eval_metric"] for r in final_results))))

dist  = Counter(r["difficulty"] for r in final_results)
total = len(final_results)
print("\nDifficulty distribution ({} book):".format(
    "open" if OPEN_BOOK else "closed"))
for level in ["Easy", "Medium", "Hard"]:
    n = dist.get(level, 0)
    print("  {:<7}: {:4d}  ({:.1f}%)".format(level, n, n / total * 100))

print("\nPer-model:")
for s in GENERATORS:
    v = [scores[(r["id"], s["name"])] for r in sample]
    print("  {:<10} EM {:>6.1%} | contains {:>6.1%}".format(
        s["name"],
        sum(x["em"] for x in v) / len(v),
        sum(x["contains"] for x in v) / len(v)))

print("\nHard rows per heritage site (top 6):")
hard = Counter(a["place"] for a in audit if a["difficulty"] == "Hard")
allp = Counter(a["place"] for a in audit)
for place, n in hard.most_common(6):
    print("  {:<44} {}/{}".format(place[:44], n, allp[place]))

print("\n--- 3 sample rows ---")
for a in audit[:3]:
    print("\n  {} [{}] votes={}".format(a["id"], a["difficulty"], a["votes"]))
    print("    cloze: {}".format(a["cloze"][:88]))
    print("    gold : {!r}".format(a["gold"]))
    for k, v in a["predictions"].items():
        print("    {:<8}: {!r}".format(k, v[:40]))

Schema check : 95 rows | wrong keys: 0 | null difficulty: 0
questions/answers altered vs input: 0
eval_metric  : {'accuracy': 95}

Difficulty distribution (closed book):
  Easy   :    0  (0.0%)
  Medium :    0  (0.0%)
  Hard   :   95  (100.0%)

Per-model:
  mistral    EM   0.0% | contains   0.0%
  llama      EM   0.0% | contains   0.0%
  gemma      EM   1.1% | contains   1.1%

Hard rows per heritage site (top 6):
  ଭାରତର ଶେଷ ସ୍ୱାଧୀନ ଦୁର୍ଗ (ଖୋର୍ଦ୍ଧାଗଡ଼)        6/6
  ଗିରିକନ୍ଦରରେ ଶ୍ରୀଚନ୍ଦ୍ରଶେଖର (କପିଳାସ)          6/6
  ତାଳସାରି                                      6/6
  ଫୁର୍ଲିଝରନ ଜଳପ୍ରପାତ                           6/6
  ଅନନ୍ୟ ପ୍ରକୃତିର ଉପହାର ଭିତରକନିକା               6/6
  ମାଳିଗୁଡ଼ା ଟନେଲ୍‍                             4/4

--- 3 sample rows ---

  heritage_000001 [Hard] votes=[0, 0, 0]
    cloze: ଅନୁଗୁଳଠାରୁ ସାତକୋଶିଆର ଦୂରତ୍ୱ ପ୍ରାୟ ______
    gold : '୯୭ କି:ମି'
    mistral : '500'
    llama   : '45 କି.ମି.'
    gemma   : '70'

  heritage_000002 [Hard] votes=[0, 0, 0]
    cloze: ଦୀର୍ଘ ______ ଏହି ଗଣ୍